<a href="https://colab.research.google.com/github/shijithpulikkal/CodingFactory/blob/main/CF10%20RFM%20Segmentation%20(Recency%2C%20Frequency%2C%20Monetary).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [3]:
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_excel('/content/drive/My Drive/Colab Notebooks/Superstore.xlsx')
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,...,Postal_Code,Region,Product_ID,Category,Sub-Category,Product_Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [5]:
df['Order_Date']= pd.to_datetime(df['Order_Date'])
ref_Date = df['Order_Date'].max()
df.head()

,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,...,Postal_Code,Region,Product_ID,Category,Sub-Category,Product_Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [15]:
rfm = df.groupby('Customer_ID').agg(
    Recency = ('Order_Date', lambda x: (ref_Date - x.max()).days),
    Frequency = ('Order_Date','count'),
    Monetary = ('Sales', 'sum')
).reset_index()

rfm

,Customer_ID,Recency,Frequency,Monetary
0,AA-10315,184,11,5563.560
1,AA-10375,19,15,1056.390
2,AA-10480,259,12,1790.512
3,AA-10645,55,18,5086.935
4,AB-10015,415,6,886.156
...,...,...,...,...
788,XP-21865,43,28,2374.658
789,YC-21895,4,8,5454.350
790,YS-21880,9,12,6720.444
791,ZC-21910,54,31,8025.707


In [16]:
rfm['R_Score'] = pd.qcut(rfm['Recency'],5,labels=[5,4,3,2,1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'),5,labels=[1,2,3,4,5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'],5,labels=[1,2,3,4,5]).astype(int)
rfm

,Customer_ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score
0,AA-10315,184,11,5563.560,2,3,5
1,AA-10375,19,15,1056.390,5,4,2
2,AA-10480,259,12,1790.512,1,3,3
3,AA-10645,55,18,5086.935,3,4,5
4,AB-10015,415,6,886.156,1,1,1
...,...,...,...,...,...,...,...
788,XP-21865,43,28,2374.658,4,5,3
789,YC-21895,4,8,5454.350,5,2,5
790,YS-21880,9,12,6720.444,5,3,5
791,ZC-21910,54,31,8025.707,3,5,5


(Note: .rank(method='first') on Frequency avoids errors from qcut when many customers share the same frequency count — a common real-data snag with this dataset.)

In [17]:
rfm['RFM_Segment'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)
rfm

,Customer_ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Segment
0,AA-10315,184,11,5563.560,2,3,5,235
1,AA-10375,19,15,1056.390,5,4,2,542
2,AA-10480,259,12,1790.512,1,3,3,133
3,AA-10645,55,18,5086.935,3,4,5,345
4,AB-10015,415,6,886.156,1,1,1,111
...,...,...,...,...,...,...,...,...
788,XP-21865,43,28,2374.658,4,5,3,453
789,YC-21895,4,8,5454.350,5,2,5,525
790,YS-21880,9,12,6720.444,5,3,5,535
791,ZC-21910,54,31,8025.707,3,5,5,355


In [18]:
rfm.sort_values('RFM_Segment', ascending = False)

,Customer_ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Segment
433,KL-16555,16,22,5016.4880,5,5,5,555
275,EP-13915,12,31,5478.0608,5,5,5,555
570,NP-18325,25,21,5529.6200,5,5,5,555
35,AI-10855,13,18,4375.7860,5,5,5,555
328,HM-14860,2,20,8236.7648,5,5,5,555
...,...,...,...,...,...,...,...,...
373,JJ-15445,483,6,709.1780,1,1,1,111
552,NB-18580,1165,2,273.8720,1,1,1,111
374,JJ-15760,753,3,195.0000,1,1,1,111
376,JK-15325,287,4,383.8120,1,1,1,111
